# MOAB Rover Survey Lab: Feature Engineering in Snowflake

## Goal
In this lab, we will engineer new features from rover-collected survey data using:
- Python UDFs
- SQL views
- AI prompts

## Raw Inputs
- Easting
- Northing
- Sensor measurement

## Engineered Features
1. Grid tile, survey unit, and subcell assignment
2. Tile-level aggregated measurement signals
3. Comparison to normal range
4. Remediation prioritization

## SCALARS Mapping
- **Simplify**: convert coordinates into tile IDs
- **Aggregate**: summarize measurements at the tile level using multi-level aggregation
- **Assess**: compare readings to expected range
- **Rank / Score**: prioritize areas for remediation

In [ ]:
%%sql -r source_data
-- Change to your user's schema
USE SCHEMA data5035.COYOTE;
USE ROLE coyote_data5035_role;
SELECT * FROM data5035.spring26.sdg_001_ra226_scandata limit 10;

## Create CONVERT_XY(...)

A Python UDF that turns raw easting/northing coordinates into grid labels (tile, survey unit, subcell).

**How it works:**
1. Subtract the known origin from each coordinate to get a relative position
2. Divide by survey unit size (32.81 ft) to find which row and column of survey units
3. Divide again by the tile grid dimensions (21 wide x 18 tall) to get the tile row and column, then convert to letters (AA, AB, etc.)
4. Use modulo to find the survey unit position inside the tile, number top-left to bottom-right
5. Divide the leftover distance inside each survey unit by the subcell size (10x10 grid) to get the subcell number, bottom-left to top-right
6. Return a JSON object with tile, su, and subcell

In [ ]:
CREATE OR REPLACE FUNCTION CONVERT_XY(
        X FLOAT,
        Y FLOAT,
        ORIGIN_X FLOAT,
        ORIGIN_Y FLOAT,
        SU_SIZE FLOAT,
        TILE_GRID_X NUMBER(38,0),
        TILE_GRID_Y NUMBER(38,0),
        SUBCELL_GRID NUMBER(38,0)
    )
    RETURNS OBJECT
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    HANDLER = 'convert_xy'
    AS 
    $$
def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):
    # step 1: get relative position from origin
    dx = x - origin_x
    dy = y - origin_y

    # step 2: find which survey unit column and row globally
    su_col_global = int(dx // su_size)
    su_row_global = int(dy // su_size)

    # step 3: find which tile by dividing by grid dimensions
    tile_col = su_col_global // tile_grid_x
    tile_row = su_row_global // tile_grid_y

    # step 4: convert tile row/col to two-letter label (AA, AB, etc.)
    tile_label = chr(ord('A') + tile_row) + chr(ord('A') + tile_col)

    # step 5: find survey unit position inside the tile
    su_col_local = su_col_global % tile_grid_x
    su_row_local = su_row_global % tile_grid_y

    # number top-left to bottom-right (flip row so top = 0)
    su_row_from_top = (tile_grid_y - 1) - su_row_local
    su_number = su_row_from_top * tile_grid_x + su_col_local + 1

    # step 6: find subcell position inside the survey unit
    subcell_size = su_size / subcell_grid
    dx_within_su = dx - su_col_global * su_size
    dy_within_su = dy - su_row_global * su_size

    # number bottom-left to top-right
    subcell_col = int(dx_within_su // subcell_size)
    subcell_row = int(dy_within_su // subcell_size)
    subcell_number = subcell_row * subcell_grid + subcell_col + 1

    return {'tile': tile_label, 'su': su_number, 'subcell': subcell_number}
    $$;

In [ ]:
%%sql -r dataframe_3
    CREATE OR REPLACE FUNCTION CONVERT_XY(X FLOAT, Y FLOAT)
    RETURNS OBJECT
    LANGUAGE SQL
    AS
    $$
        SELECT CONVERT_XY(
            X, Y,
            2180160.0001,
            6660000.0000,
            32.81,
            21,
            18,
            10
        )
    $$;

In [ ]:
%%sql -r dataframe_4
select convert_xy(2180160.0001, 6660000.0000);

In [ ]:
%%sql -r dataframe_5
select convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000);

In [ ]:
%%sql -r dataframe_6
select convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)

## Compute Layered Averages

Uses `convert_xy()` to assign each reading to a tile, survey unit, and subcell, then groups by all three and computes the average reading per subcell. Results are sorted by tile, survey unit, and subcell number.

In [ ]:
%%sql -r dataframe_7
SELECT 
    convert_xy(easting, northing) AS coordinates,
    coordinates:tile::STRING AS tile,
    coordinates:su::INTEGER AS su,
    coordinates:subcell::INTEGER AS subcell,
    
    avg(reading)
FROM 
    data5035.spring26.sdg_001_ra226_scandata
GROUP BY ALL
ORDER BY 2, 3, 4;

## Compare to Reference Ranges

Checks each subcell's average Ra-226 reading against regulatory thresholds.

**How it works:**
1. Compute the per-subcell average in the `subcell_avgs` CTE
2. Classify each subcell as OK (< 5), Warning (5–7.4), or Alarm (≥ 7.4 pCi/g)
3. Roll up a summary count per tile showing how many subcells fall into each status

In [ ]:
-- step 1: compute per-subcell average reading
WITH subcell_avgs AS (
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        AVG(reading)                     AS avg_reading
    FROM data5035.spring26.sdg_001_ra226_scandata
    GROUP BY ALL
),
-- step 2: classify each subcell as OK, Warning, or Alarm
classified AS (
    SELECT
        tile,
        su,
        subcell,
        avg_reading,
        CASE
            WHEN avg_reading < 5   THEN 'OK'
            WHEN avg_reading < 7.4 THEN 'Warning'
            ELSE 'Alarm'
        END AS status
    FROM subcell_avgs
)
-- step 3: roll up counts per tile
SELECT
    tile,
    COUNT_IF(status = 'OK')              AS ok_count,
    COUNT_IF(status = 'Warning')         AS warning_count,
    COUNT_IF(status = 'Alarm')           AS alarm_count,
    ROUND(AVG(avg_reading), 4)           AS tile_avg_reading,
    ROUND(MAX(avg_reading), 4)           AS tile_max_reading,
    COUNT(*)                             AS total_subcells
FROM classified
GROUP BY tile
ORDER BY alarm_count DESC, warning_count DESC

## Combine — Neighboring Tile Change

Measures how much the average reading jumps between neighboring tiles.

**How it works:**
1. Compute the average reading per tile
2. Extract the row and column letters from each tile label
3. Self-join tiles where row and column letters are within one step of each other (up, down, left, right, diagonal)
4. Subtract the two tile averages and take the absolute value
5. Sort by largest change first — big jumps may indicate hotspot edges or data issues

In [ ]:
-- step 1: get tile label for each reading
WITH grid AS (
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        reading
    FROM data5035.spring26.sdg_001_ra226_scandata
),
-- step 2: compute average per tile and extract row/col letters
tile_avgs AS (
    SELECT
        tile,
        SUBSTR(tile, 1, 1) AS tile_row,
        SUBSTR(tile, 2, 1) AS tile_col,
        AVG(reading) AS tile_avg
    FROM grid
    GROUP BY tile
)
-- step 3: self-join neighbors (within 1 step) and compute absolute difference
SELECT
    a.tile,
    ROUND(a.tile_avg, 2) AS tile_avg,
    b.tile AS neighbor_tile,
    ROUND(b.tile_avg, 2) AS neighbor_avg,
    ROUND(ABS(a.tile_avg - b.tile_avg), 2) AS tile_change
FROM tile_avgs a
JOIN tile_avgs b
    ON ABS(ASCII(a.tile_row) - ASCII(b.tile_row)) <= 1
    AND ABS(ASCII(a.tile_col) - ASCII(b.tile_col)) <= 1
    AND a.tile != b.tile
ORDER BY tile_change DESC;

## Rank Within Region

Ranks areas from highest to lowest reading within their parent region.

**How it works:**
1. Compute average reading per subcell in `subcell_avgs`
2. Compute average reading per survey unit in `su_avgs`
3. Join the two, then use RANK() windows to assign:
   - `subcell_rank_in_su` — which subcell is highest within its survey unit
   - `su_rank_in_tile` — which survey unit is highest within its tile
4. Rank 1 = most contaminated in that local area

In [ ]:
-- step 1: compute average reading per subcell
WITH subcell_avgs AS (
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        AVG(reading)                     AS subcell_avg
    FROM data5035.spring26.sdg_001_ra226_scandata
    GROUP BY ALL
),
-- step 2: compute average per survey unit
su_avgs AS (
    SELECT tile, su, AVG(subcell_avg) AS su_avg
    FROM subcell_avgs
    GROUP BY tile, su
)
-- step 3: rank subcells within SU and SUs within tile
SELECT
    s.tile,
    s.su,
    s.subcell,
    ROUND(s.subcell_avg, 2) AS subcell_avg,
    RANK() OVER (PARTITION BY s.tile, s.su ORDER BY s.subcell_avg DESC) AS subcell_rank_in_su,
    ROUND(u.su_avg, 2) AS su_avg,
    RANK() OVER (PARTITION BY s.tile ORDER BY u.su_avg DESC) AS su_rank_in_tile
FROM subcell_avgs s
JOIN su_avgs u ON s.tile = u.tile AND s.su = u.su
ORDER BY s.tile, s.su, subcell_rank_in_su;

## Stability Score

Measures how consistent the readings are within each subcell.

**How it works:**
1. Group all readings by tile, survey unit, and subcell
2. Compute the standard deviation and count of readings
3. Apply the formula: `1 / (1 + stddev)` — low spread gives a score near 1 (trustworthy), high spread gives a score near 0 (unreliable)
4. Sort lowest stability first so unreliable areas appear at the top

In [ ]:
-- step 1: assign each reading to tile/su/subcell
WITH subcell_data AS (
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        reading
    FROM data5035.spring26.sdg_001_ra226_scandata
)
-- step 2: compute count, average, stddev, and stability score per subcell
SELECT
    tile,
    su,
    subcell,
    COUNT(reading) AS num_readings,
    ROUND(AVG(reading), 2) AS subcell_avg,
    ROUND(STDDEV(reading), 4) AS stddev_reading,
    ROUND(1.0 / (1.0 + COALESCE(STDDEV(reading), 0)), 4) AS stability_score
FROM subcell_data
GROUP BY tile, su, subcell
ORDER BY stability_score ASC;

## Prioritize

Once we've build everything out, let's use AI to make some recommendations on remediation priorities.

In [ ]:
-- step 1: compute average, stddev, and count per subcell
WITH subcell_avgs AS (
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        AVG(reading)                     AS avg_reading,
        STDDEV(reading)                  AS stddev_reading,
        COUNT(reading)                   AS num_readings
    FROM data5035.spring26.sdg_001_ra226_scandata
    GROUP BY ALL
),
-- step 2: roll up to survey unit average
su_avgs AS (
    SELECT tile, su, AVG(avg_reading) AS su_avg
    FROM subcell_avgs
    GROUP BY tile, su
),
-- step 3: roll up to tile average
tile_avgs AS (
    SELECT tile, AVG(avg_reading) AS tile_avg
    FROM subcell_avgs
    GROUP BY tile
),
-- step 4: join all levels, classify each as OK/Warning/Alarm, add stability score
combined AS (
    SELECT
        s.tile,
        s.su,
        s.subcell,
        ROUND(s.avg_reading, 2) AS subcell_avg,
        CASE WHEN s.avg_reading < 5 THEN 'OK' WHEN s.avg_reading < 7.4 THEN 'Warning' ELSE 'Alarm' END AS subcell_status,
        ROUND(u.su_avg, 2) AS su_avg,
        CASE WHEN u.su_avg < 5 THEN 'OK' WHEN u.su_avg < 7.4 THEN 'Warning' ELSE 'Alarm' END AS su_status,
        ROUND(t.tile_avg, 2) AS tile_avg,
        CASE WHEN t.tile_avg < 5 THEN 'OK' WHEN t.tile_avg < 7.4 THEN 'Warning' ELSE 'Alarm' END AS tile_status,
        ROUND(1.0 / (1.0 + COALESCE(s.stddev_reading, 0)), 4) AS stability_score,
        s.num_readings
    FROM subcell_avgs s
    JOIN su_avgs u ON s.tile = u.tile AND s.su = u.su
    JOIN tile_avgs t ON s.tile = t.tile
),
-- step 5: pick the top 10 worst alarm subcells
alarm_subcells AS (
    SELECT tile, su, subcell, subcell_avg, subcell_status, su_avg, su_status, tile_avg, tile_status, stability_score, num_readings,
        'Top Alarm Subcell' AS finding_category
    FROM combined WHERE subcell_status = 'Alarm'
    ORDER BY subcell_avg DESC
    LIMIT 10
),
-- step 6: pick the top 10 warning or at-risk subcells
warning_areas AS (
    SELECT tile, su, subcell, subcell_avg, subcell_status, su_avg, su_status, tile_avg, tile_status, stability_score, num_readings,
        'Warning / At-Risk Tile' AS finding_category
    FROM combined
    WHERE subcell_status = 'Warning' OR (tile_status IN ('Alarm','Warning') AND subcell_status = 'OK')
    ORDER BY subcell_avg DESC
    LIMIT 10
),
-- step 7: pick the top 10 least stable subcells (unreliable data)
low_stability AS (
    SELECT tile, su, subcell, subcell_avg, subcell_status, su_avg, su_status, tile_avg, tile_status, stability_score, num_readings,
        'Low Stability - Needs Re-Survey' AS finding_category
    FROM combined WHERE stability_score < 0.15
    ORDER BY stability_score ASC
    LIMIT 10
),
-- step 8: combine all three finding groups into one set
all_findings AS (
    SELECT * FROM alarm_subcells
    UNION ALL
    SELECT * FROM warning_areas
    UNION ALL
    SELECT * FROM low_stability
)
-- step 9: display each finding with an AI recommendation from Cortex
SELECT
    finding_category,
    tile,
    su,
    subcell,
    subcell_avg,
    subcell_status,
    su_avg,
    su_status,
    tile_avg,
    tile_status,
    stability_score,
    num_readings,
    SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-70b',
        'AS an environmental remediation specialist. Given this Ra-226 finding: '
        || 'Category: ' || finding_category
        || ', Tile: ' || tile || ', SU: ' || su || ', Subcell: ' || subcell
        || ', Subcell Avg: ' || subcell_avg || ' pCi/g (' || subcell_status || ')'
        || ', SU Avg: ' || su_avg || ' pCi/g (' || su_status || ')'
        || ', Tile Avg: ' || tile_avg || ' pCi/g (' || tile_status || ')'
        || ', Stability Score: ' || stability_score
        || ', Num Readings: ' || num_readings
        || '. Thresholds: OK < 5, Warning 5-7.4, Alarm >= 7.4 pCi/g. '
        || 'In 1-2 sentences, recommend a specific remediation action for this location.'
    ) AS ai_recommendation
FROM all_findings
ORDER BY
    CASE finding_category
        WHEN 'Top Alarm Subcell' THEN 1
        WHEN 'Warning / At-Risk Tile' THEN 2
        WHEN 'Low Stability - Needs Re-Survey' THEN 3
    END,
    subcell_avg DESC;